In [1]:
import time
import numpy as np
from scipy import signal
from pynq import Overlay, allocate
from rtlsdr import RtlSdr

ol = Overlay("./base.bit")

/usr/local/share/pynq-venv/lib/python3.8/site-packages/pynq/bitstream.py:151: UserWarning: The provided name './base.bit' resulted in multiple possible matches:
 - /home/xilinx/jupyter_notebooks/audio/base.bit
 - /usr/local/share/pynq-venv/lib/python3.8/site-packages/pynq/overlays/./base/./base.bit
The first entry of this list, '/home/xilinx/jupyter_notebooks/audio/base.bit', will be used, please provide the full path in case your target file was a different one in this list.
  warnings.warn(msg, UserWarning)


In [2]:
# check IP contents in bitstream
print(ol.ip_dict.keys())

dma = ol.rtl_sdr.axi_dma_0

dict_keys(['rtl_sdr/axi_dma_0', 'switches_gpio', 'btns_gpio', 'rgbleds_gpio', 'leds_gpio', 'audio_codec_ctrl_0', 'video/axi_vdma_0', 'video/axi_dynclk_0', 'video/v_tc_0', 'simple_io_0', 'axi_quad_spi_0', 'lcd_gpio', 'ps7_0'])


/usr/local/share/pynq-venv/lib/python3.8/site-packages/pynq/overlay.py:681: UserWarning: Interrupt mm2s_introut not created: Could not find UIO device for interrupt pin for IRQ number 64
  warnings.warn('Interrupt {} not created: {}'.format(
/usr/local/share/pynq-venv/lib/python3.8/site-packages/pynq/overlay.py:681: UserWarning: Interrupt s2mm_introut not created: Could not find UIO device for interrupt pin for IRQ number 65
  warnings.warn('Interrupt {} not created: {}'.format(


In [3]:
len_in = 1024 * 1500
len_out = int(len_in / 40)

resize = 1 / 40

In [4]:
# make input buffer and output buffer for AXI DMA
input_buffer = allocate(shape=(len_in,), dtype=np.uint16)
output_buffer = allocate(shape=(len_out,), dtype=np.int16)

print(f"Input Buffer Address: {hex(input_buffer.device_address)}")
print(f"Output Buffer Address: {hex(output_buffer.device_address)}")

Input Buffer Address: 0x16900000
Output Buffer Address: 0x16860000


In [5]:
# connect and configure RTL-SDR
sdr = RtlSdr()

fc = 99750000
fs = 1920000
ppm = 60

sdr.sample_rate = fs
sdr.center_freq = fc
sdr.freq_correction = ppm
sdr.gain = "auto"

In [6]:
# increase usb memory buffer
!echo 0 > /sys/module/usbcore/parameters/usbfs_memory_mb

In [7]:
from IPython.display import Audio
import nest_asyncio
import asyncio

## Audio IP status monitor (register 0x10)

รัน 2 เซลล์ถัดไป **ก่อน** เซลล์ streaming แล้วรันเซลล์ report **หลัง** streaming จบ

| bit | ความหมาย |
|---|---|
| 0 | data_rdy (เดิม) |
| 1 | playing (FIFO prefill ถึง 64 แล้ว) |
| 2 | fifo_full |
| 3 | fifo_empty |
| 4 | tready |
| 5 | tvalid (live) |
| 6 | underflow (sticky) |
| 7 | stream_seen (sticky: เคยรับ sample เข้า IP) |
| 16:8 | fifo_count (0..256) |
| 31:17 | rx_cnt (นับ sample ที่รับเข้า, 15-bit วนรอบ) |

เขียน `0x80000000` ที่ 0x10 เพื่อเคลียร์ rx_cnt / stream_seen / underflow

In [ ]:
import time, threading

audio_ip = ol.audio_codec_ctrl_0
STATUS_OFF = 0x10
CLR_STATUS = 0x80000000   # bit31 = 1 -> clear rx_cnt / stream_seen / underflow

def read_status():
    v = audio_ip.mmio.read(STATUS_OFF)
    return {
        "raw": v,
        "data_rdy": v & 1,
        "playing": (v >> 1) & 1,
        "full": (v >> 2) & 1,
        "empty": (v >> 3) & 1,
        "tready": (v >> 4) & 1,
        "tvalid": (v >> 5) & 1,
        "underflow": (v >> 6) & 1,
        "stream_seen": (v >> 7) & 1,
        "fifo_count": (v >> 8) & 0x1FF,
        "rx_cnt": (v >> 17) & 0x7FFF,
    }

def clear_status():
    audio_ip.mmio.write(STATUS_OFF, CLR_STATUS)

def print_status():
    s = read_status()
    print(f"raw = 0x{s['raw']:08X}")
    for k, v in s.items():
        if k != "raw":
            print(f"  {k:12s}: {v}")

print_status()

In [ ]:
class StatusMonitor:
    # poll register 0x10 in a background thread while streaming runs
    def __init__(self, period=0.02):
        self.period = period
        self.history = []
        self._run = False
        self._th = None

    def _loop(self):
        while self._run:
            s = read_status()
            v = audio_ip.mmio.read(0x8) & 0xFFFFFF      # DataTx_L (24-bit signed)
            if v & 0x800000:
                v -= 0x1000000
            s["tx_l"] = v
            self.history.append((time.perf_counter(), s))
            time.sleep(self.period)

    def start(self):
        self.stop()
        clear_status()
        self.history = []
        self._run = True
        self._th = threading.Thread(target=self._loop, daemon=True)
        self._th.start()

    def stop(self):
        self._run = False
        if self._th is not None:
            self._th.join()
            self._th = None

mon = StatusMonitor(period=0.02)
mon.start()
print("status monitor started (20 ms polling)")

In [8]:
nest_asyncio.apply()

flag = 0

it_num = 10

async def streaming():
    it = 0
    async for data in sdr.stream(num_samples_or_bytes=len_in*2, format='bytes', loop=None):
        global flag
        
        raw_uint8 = np.frombuffer(data, dtype=np.uint8)
        
        raw_int8 = (raw_uint8.astype(np.int16) - 128).astype(np.int8)
        
        packed_samples = raw_int8.view(np.uint16)
        
        np.copyto(input_buffer, packed_samples)
        
        if flag == 1:
            dma.sendchannel.wait()
            dma.recvchannel.wait()
        else:
            flag = 1
            
        dma.sendchannel.transfer(input_buffer)
        dma.recvchannel.transfer(output_buffer)
        
        audio_data = output_buffer.astype(np.float32) / 32768.0
        
        display(Audio(audio_data, autoplay=True, rate=48000, normalize=False))
        
        it = it + 1
        print("sample %d" % it)
        if it == it_num:
            print("Stopping stream...")
            await sdr.stop()
            sdr.close()
            break
        
asyncio.run(streaming())

sample 1


sample 2


sample 3


sample 4


sample 5


sample 6


sample 7


sample 8


sample 9


sample 10
Stopping stream...


### Streaming status report

In [ ]:
def report(mon):
    mon.stop()
    h = mon.history
    if len(h) < 2:
        print("no samples collected")
        return

    n = len(h)
    dur = h[-1][0] - h[0][0]
    total = sum((b["rx_cnt"] - a["rx_cnt"]) & 0x7FFF
                for (_, a), (_, b) in zip(h, h[1:]))
    pct = lambda k: 100.0 * sum(s[k] for _, s in h) / n
    counts = [s["fifo_count"] for _, s in h]
    last = h[-1][1]

    print(f"polls            : {n} over {dur:.2f} s")
    print(f"samples received : {total}  ({total / dur / 1000:.2f} kS/s, codec reads 48.83 kS/s)")
    print(f"stream_seen      : {last['stream_seen']}")
    print(f"underflow (stick): {last['underflow']}")
    print(f"playing          : {pct('playing'):.1f} % of time")
    print(f"tvalid           : {pct('tvalid'):.1f} %   tready: {pct('tready'):.1f} %")
    print(f"fifo_count       : min {min(counts)}  max {max(counts)}")
    tx = [s["tx_l"] for _, s in h]
    print(f"DataTx_L (0x8)   : min {min(tx)}  max {max(tx)}  distinct {len(set(tx))}")

    print("--- diagnosis ---")
    if not last["stream_seen"]:
        print("no sample ever reached the IP -> check S2MM armed (broadcaster stalls), "
              "MM2S feeding IQ, HLS filters running")
    elif max(counts) < 64:
        print("samples arrive but FIFO never reaches 64 -> stream too slow/bursty")
    elif pct("playing") == 0:
        print("FIFO filled but never playing -> unexpected, check reset/clock")
    else:
        print("stream reaches the IP and is read at 48 kHz")
        if len(set(tx)) < 10:
            print("-> DataTx_L is (almost) constant: audio data is silent / DC (check FM demod output)")
        else:
            print("-> DataTx_L is changing: data reaches I2S output")
        print("-> if still silent: codec path (I2C config / volume / mute)")
        if last["underflow"]:
            print("-> underflow seen: stream has gaps (DMA/Python restart between blocks)")

report(mon)

In [9]:
# delete input buffer and output buffer
del input_buffer, output_buffer

In [12]:
from pynq import Overlay
from pathlib import Path

WAV_FILE = "sine_440.wav"

AUDIO_BASE = 0x43C00000


# ==========================================
# Access Audio IP
# ==========================================

pAudio = ol.audio_codec_ctrl_0

print("[INFO] Audio Base Address:",
      hex(pAudio.mmio.base_addr))

assert pAudio.mmio.base_addr == AUDIO_BASE

[INFO] Audio Base Address: 0x43c00000


In [13]:

# ==========================================
# Configure Audio Codec
# ==========================================

print("[INFO] Configuring Audio Codec...")

pAudio.configure(
    sample_rate=48000,
    iic_index=1,
    uio_name="audio-codec-ctrl"
)

print("[INFO] UIO Index:", pAudio.uio_index)
print("[INFO] I2C Index:", pAudio.iic_index)

# ==========================================
# Set Volume
# ==========================================

pAudio.set_volume(60)

# ==========================================
# Load WAV
# ==========================================

if not Path(WAV_FILE).exists():
    raise FileNotFoundError(WAV_FILE)

print("[INFO] Loading WAV...")

pAudio.load(WAV_FILE)

print("[INFO] Sample Rate:", pAudio.sample_rate)
print("[INFO] Sample Length:", pAudio.sample_len)
print("[INFO] Buffer Size:", len(pAudio.buffer))

# ==========================================
# Playback
# ==========================================

print("[INFO] Playing Audio...")

pAudio.play()

print("[INFO] Playback completed.")

[INFO] Configuring Audio Codec...
[INFO] UIO Index: 0
[INFO] I2C Index: 1
[INFO] Loading WAV...
[INFO] Sample Rate: 44100
[INFO] Sample Length: 88200
[INFO] Buffer Size: 176400
[INFO] Playing Audio...
[INFO] Playback completed.
